# Florence-2-large — DIMER E2E handwritten-line OCR adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/florence2-vision-language-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/florence2-vision-language-pipeline/blob/main/tutorials/florence2_vision_language_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-florence--community%2FFlorence--2--large-ffcc4d?style=flat)](https://huggingface.co/florence-community/Florence-2-large) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FFlorence--2--large-181717?style=flat&logo=huggingface&logoColor=white)](https://huggingface.co/microsoft/Florence-2-large) [![arXiv](https://img.shields.io/badge/arXiv-2311.06242-b31b1b.svg)](https://arxiv.org/abs/2311.06242)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** prompt-selected vision-language tasks (caption, detection and OCR demonstrated) and bounded supervised fine-tuning of the `<OCR>` task's last decoder layers on transcribed text lines, using the pinned Florence-2-large weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/florence2_vision_language_pipeline/`, at revision `d9ffce47a911`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `4271c66b88cdbc05735372ec13b2360108de5317` (~1559 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `florence-community/Florence-2-large` snapshot (a 1.54 GB `model.safetensors`; no pickle is opened anywhere), fetches the first eight row groups of the Belfort-line test shard from the Hugging Face Hub at an immutable revision with HTTPS range requests (about 44 MB; each row group refused on any SHA-256 or byte-total mismatch), validates the 800 line records and splits them by line into 600 / 60 / 140, runs three capabilities (`<CAPTION>`, `<OD>`, `<OCR>`) on a synthetic drawing through the inference contract with a combined input manifest and a rejection probe, measures the frozen model's `<OCR>` character and word error rates over the 140 held-out lines beside an empty-string and a constant-transcript baseline, runs a bounded fine-tuning of the last four BART decoder layers on cached encoder outputs with validation-CER epoch selection, scores the held-out lines again, re-runs six held-out lines and the three capabilities on the drawing with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify transcript parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a Tesla T4 the default path took about @P:T4_TOTAL_MIN@ minutes of cell time (six epochs @P:T4_ADAPT_S@ s, frozen scoring of 140 lines @P:T4_FROZEN_S@ s); a CUDA runtime is used automatically when present, and **a CPU runtime is not practical for the default path** (beam-search decoding of some 800 lines plus 600 cached forwards of a 777M-parameter model).

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip of line images plus a `transcripts.csv` (`file`, `text`, optional `id`; one row per image, at least eight images). The records pass through the same validation, image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the Belfort sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

Florence-2 is one sequence-to-sequence model whose behaviour is selected by a **task prompt token**: the image is resized to 768 × 768 by the processor (aspect ratio is not preserved), encoded by a DaViT vision tower into 577 visual tokens that a BART encoder reads together with the prompt, and a 12-layer BART decoder generates text that the processor parses per task — plain text for captions and OCR, boxes plus labels in input-pixel coordinates for region tasks (776,505,344 parameters in all, published under the **MIT** licence). Decoding is **deterministic beam search** (`num_beams` = 3 from the snapshot generation config, `do_sample=False`). The output carries **no score**: captions and transcripts are plain generated text, boxes have no confidence.

What this notebook adds to inference is **adaptation of one task on transcribed lines**. Florence-2's `<OCR>` was trained on printed and scene text; nineteenth-century French council minutes in cursive handwriting — the Belfort-line dataset — are far outside that distribution, and on them the frozen model reads nothing: it emits a dash or an empty string for almost every line, a character error rate of **@P:FROZEN_CER@** on the 140 held-out lines (the build record's Tesla T4 figure), the empty baseline's 1.0 in all but name. So the honest question is narrow: does a bounded fine-tuning of the last four decoder layers on 600 transcribed lines move the held-out **CER** and **WER** on a line-disjoint test split past two **non-adapted baselines** and the frozen model — and what does it do to the other tasks the same decoder serves? Nothing here is a claim about your documents or your script: it is one seeded split of one small labelled set.

**Snapshot note:** the pin is the community "official transformers converted checkpoint" (`florence-community/Florence-2-large`), loadable by native `transformers` classes with `trust_remote_code=False`; the original `microsoft/Florence-2-large` snapshot needs remote code and was rejected (see the weight provenance document). The pinned revision ships `model.safetensors` (a 12-file manifest with the tokenizer files) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the processor or the model is constructed, and the loader refuses a checkpoint whose weights do not map cleanly onto the native architecture. The pipeline loads the checkpoint in **float32 on every device**: the adapter is trained in float32 and overlays without a cast, and CPU, Tesla-class and consumer GPUs then run the same arithmetic.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled line set with its transcripts, validate it and split it by line without leakage; run three task tokens on a synthetic drawing through the public API and read each output contract correctly (generated text, no score, boxes without confidence, error rates against text you drew are not a benchmark); measure the frozen `<OCR>` corpus CER and WER beside two non-adapted baselines; run a bounded fine-tuning with the model's own sequence-to-sequence loss, explicit hyperparameters and validation-based epoch selection; evaluate on a line-disjoint test split; look at the adapted transcripts next to the frozen ones and the references, and at what the other two capabilities do after the shared decoder was tuned; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** the other task tokens in `TASKS` are exposed but not demonstrated — `<DETAILED_CAPTION>`, `<MORE_DETAILED_CAPTION>`, `<DENSE_REGION_CAPTION>`, `<REGION_PROPOSAL>`, `<OCR_WITH_REGION>` and `<CAPTION_TO_PHRASE_GROUNDING>` (the only task that takes a `text_input`); segmentation of any kind, open-vocabulary detection, visual question answering, confidence scores for boxes or text, fine-tuning of the vision tower, the projector, the encoder, the embeddings or the first eight decoder layers, fine-tuning of any task but `<OCR>`, evaluation on an OCR benchmark proper (only one seeded 800-line sample is scored here), and any claim that French cursive minutes stand in for your documents. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime with a CUDA GPU (Google Colab or Kaggle GPU, Python 3.12). The default path uses CUDA automatically when present. Generation is batched for the corpus stages — every `<OCR>` prompt is the same 587 tokens (577 image tokens plus the task token), so a batch needs no padding — and the build record measured @P:T4_FROZEN_S@ s to score 140 lines with 3-beam search and @P:T4_ADAPT_S@ s for the six epochs (caching the encoder outputs for 600 lines took @P:T4_CACHE_S@ s) on a Tesla T4, about @P:T4_TOTAL_MIN@ minutes of cell time for the whole path; a CPU runtime would take hours. The pinned `torch==2.14.0` install and the 1.54 GB checkpoint are the large downloads of the run; the row groups are about 44 MB.
- **Knowledge:** basic Python and PIL; what a sequence-to-sequence model's task prompt and beam search are; what a bounding box in pixel coordinates is; what character and word error rate measure and why they are not capped at 1; why a self-drawn image is a plumbing check while a held-out split of one labelled set is a measurement of that set only.
- **Data contract:** records are `{id, image, text}` — `image` a PIL image (or a file decodable by Pillow) with sides within 1..16,384 px and at most 4096² pixels, `text` its transcript (1..512 characters after whitespace runs are collapsed; case and punctuation kept). Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 8..5,000 records; splitting de-duplicates by decoded pixels so no image lands in two splits. BYOD accepts one zip (or directory) of images plus a `transcripts.csv` in the layout named above.
- **Validation is structural, not semantic:** every image is decoded and every transcript checked for length, but nothing checks that a transcript says what its image shows — a mislabelled set is fine-tuned on without complaint.
- **Expected output:** a "slow image processor" notice from `transformers` is expected and harmless. A `RuntimeError: checkpoint does not match the native Florence-2 architecture` means the staged weights are not the pinned converted checkpoint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads eight row groups of `default/test/0000.parquet` from `https://huggingface.co/datasets/Teklia/Belfort-line/resolve/<revision>/` at the immutable parquet-conversion revision `c4a74bbd…` with HTTPS range requests (the parquet footer plus about 44 MB of row-group bytes out of a 210 MB shard), each row group pinned by SHA-256 and byte total in the carried `samples.py` and refused on any mismatch. Belfort-line is published under the MIT licence (Teklia; Tarride et al. 2023); nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `florence-community/Florence-2-large` snapshot (~1559 MB in total) at revision `4271c66b88cd…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'florence2-vision-language-pipeline',
    'repository_revision': 'd9ffce47a91188aeebdc4bc978a81a27e481a4e5',
    'embedded_module': 'src/florence2_vision_language_pipeline/pipeline.py',
    'embedded_modules': ['src/florence2_vision_language_pipeline/pipeline.py', 'src/florence2_vision_language_pipeline/metrics.py', 'src/florence2_vision_language_pipeline/samples.py'],
    'module_sha256': '1da327ba17a513d5957c09faf02f8bf057ffc73f252de5c8ade6745e396199f6',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/florence2_vision_language_pipeline/` @ `d9ffce47a911`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/florence2_vision_language_pipeline/pipeline.py`

In [ ]:
"""Prompt-driven vision-language tasks with the pinned ``florence-community/Florence-2-large`` snapshot, plus the
adaptation contract for the ``<OCR>`` task: corpus-level evaluation on labelled text lines, bounded fine-tuning of the
last decoder layers on cached encoder outputs, and a verified adapter artifact.

The class loads weights only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly
allowed, from the Hugging Face Hub at the pinned revision, through the native ``transformers`` Florence-2
classes with ``trust_remote_code=False``. Pin history: the original ``microsoft/Florence-2-large`` pin was
rejected on 2026-09-12 because loading it requires executing custom code bundled in the model repository.
"""

# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

import hashlib
import json
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "florence-community/Florence-2-large"
MODEL_REVISION = "4271c66b88cdbc05735372ec13b2360108de5317"
MODEL_LICENSE = "mit"
MODEL_KEY = "florence-2-large-community"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

# Task prompts documented upstream for Florence-2; the last one needs a caption as text input.
TASKS_WITHOUT_TEXT = (
    "<CAPTION>",
    "<DETAILED_CAPTION>",
    "<MORE_DETAILED_CAPTION>",
    "<OD>",
    "<DENSE_REGION_CAPTION>",
    "<REGION_PROPOSAL>",
    "<OCR>",
    "<OCR_WITH_REGION>",
)
TASKS_WITH_TEXT = ("<CAPTION_TO_PHRASE_GROUNDING>",)
TASKS = TASKS_WITHOUT_TEXT + TASKS_WITH_TEXT

# The processor resizes every image to 768x768 regardless (preprocessor_config.json), so the side and pixel
# ceilings only guard memory while decoding and resizing (a 9,000 px wide text line is fine, a 4096x4096 page is
# the largest area accepted).
MAX_IMAGE_SIDE = 16_384
MAX_IMAGE_PIXELS = 4096 * 4096
MIN_IMAGE_SIDE = 1
MAX_TEXT_CHARS = 1000  # characters of caption text accepted for phrase grounding
MAX_NEW_TOKENS = 1024  # hard ceiling for `max_new_tokens` (upstream examples use 1024)
DEFAULT_MAX_NEW_TOKENS = 256
DEFAULT_LINE_MAX_NEW_TOKENS = 128  # the corpus stages' budget per text line
NUM_BEAMS = 3  # snapshot generation_config.json; decoding is deterministic beam search (do_sample=False)
OCR_PROMPT_TOKENS = 587  # 577 image tokens + the <OCR> prompt; identical for every image
# Model facts (measured on the pinned snapshot; tests pin them).
PARAMETER_COUNT = 776_505_344
DECODER_LAYERS = 12
TRAINABLE_LAYERS = 4  # the last decoder layers + the decoder's embedding layer norm are the adapter
ADAPTER_PARAMETERS = 67_188_736
# Adaptation contract.
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = "1.0"
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 5_000
EVAL_BATCH_SIZE = 8
CACHE_BATCH_SIZE = 4  # images per frozen forward while caching encoder outputs
VISION_BATCH_SIZE = 4  # images per vision-tower forward (bounds its activation memory)
GRAD_CLIP = 1.0
_TRAINABLE_FIRST_LAYER = DECODER_LAYERS - TRAINABLE_LAYERS
_TRAINABLE_PREFIXES = tuple(f"model.language_model.decoder.layers.{i}." for i in range(_TRAINABLE_FIRST_LAYER, DECODER_LAYERS)) + ("model.language_model.decoder.layernorm_embedding.",)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _weight_digest(root: Path) -> str | None:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        return None
    with open(manifest_path, encoding="utf-8") as handle:
        entries = json.load(handle).get("files", [])
    return next((e["sha256"] for e in entries if e["path"] == WEIGHTS_FILE), None)


def normalise_text(text: str) -> str:
    """The transcript form the corpus measures use: whitespace runs collapsed to one space, ends stripped."""
    return " ".join(str(text).split())


def edit_distance(reference: Sequence[Any], hypothesis: Sequence[Any]) -> int:
    """Levenshtein distance (insertions + deletions + substitutions, unit cost) between two sequences."""
    previous = list(range(len(hypothesis) + 1))
    for row_index, ref_item in enumerate(reference, 1):
        current = [row_index]
        for column_index, hyp_item in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[column_index] + 1, previous[column_index - 1] + (ref_item != hyp_item)))
        previous = current
    return previous[-1]


def character_error_rate(reference: str, hypothesis: str) -> float:
    """Character-level Levenshtein distance over reference length; for OCR against a known transcript."""
    if not isinstance(reference, str) or not isinstance(hypothesis, str):
        raise TypeError("reference and hypothesis must be str")
    if not reference:
        return 0.0 if not hypothesis else 1.0
    return edit_distance(reference, hypothesis) / len(reference)


def validate_image(image: Any) -> Image.Image:
    """The image contract every task and every dataset record shares; returns the RGB image."""
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side outside {MIN_IMAGE_SIDE}..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    if width * height > MAX_IMAGE_PIXELS:
        raise ValueError(f"image area {width * height} px > MAX_IMAGE_PIXELS {MAX_IMAGE_PIXELS}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one PIL.Image.Image (any mode, converted to RGB) plus one task prompt from TASKS; the tasks "
        "in TASKS_WITH_TEXT additionally require a caption string as text_input"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "image_pixels_max": MAX_IMAGE_PIXELS,
    "tasks": list(TASKS),
    "tasks_requiring_text_input": list(TASKS_WITH_TEXT),
    "text_input_chars": [1, MAX_TEXT_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "num_beams": (
        f"positive int; the snapshot's generation_config default is NUM_BEAMS={NUM_BEAMS} and decoding "
        "is deterministic beam search (do_sample=False)"
    ),
    "preprocessing": (
        "image converted to RGB; the processor resizes it to exactly 768x768 (bicubic, ImageNet "
        "mean/std), so aspect ratio is not preserved and nothing is cropped; region outputs are mapped "
        "back to input pixel coordinates. The prompt sent to the model is the task token followed by "
        "text_input when the task takes one."
    ),
}

# Which capability families have an intrinsic metric in this repository, and what the others need.
_OCR_TASK = "<OCR>"
_NEEDS: dict[str, str] = {
    "caption": (
        "reference captions for the same images plus a caption metric (for example CIDEr or SPICE), or "
        "human adequacy ratings; this repository ships neither the references nor a caption metric"
    ),
    "region": (
        "annotated boxes for the same images and the caller's own matching/mean-average-precision code; "
        "Florence-2 emits no per-box score, so there is also nothing to calibrate or threshold"
    ),
    "ocr": (
        "a known transcript for the image, passed as the reference, so character_error_rate can be "
        "computed"
    ),
    "grounding": (
        "annotated boxes for the phrases in the supplied caption and the caller's own matching code; no "
        "grounding metric ships with this repository"
    ),
}
_TASK_FAMILY: dict[str, str] = {
    "<CAPTION>": "caption",
    "<DETAILED_CAPTION>": "caption",
    "<MORE_DETAILED_CAPTION>": "caption",
    "<OD>": "region",
    "<DENSE_REGION_CAPTION>": "region",
    "<REGION_PROPOSAL>": "region",
    "<OCR>": "ocr",
    "<OCR_WITH_REGION>": "region",
    "<CAPTION_TO_PHRASE_GROUNDING>": "grounding",
}
_SCORE_SEMANTICS = (
    "Florence-2 emits no probability or confidence: captions and OCR are plain generated text, and "
    "region tasks return boxes with labels and no per-box score, so there is nothing to threshold or "
    "calibrate. Decoding is deterministic beam search, not a likelihood estimate."
)


def _check_inputs(
    image: Any, task: Any, text_input: Any, max_new_tokens: Any, num_beams: Any
) -> Image.Image:
    """Raise TypeError/ValueError naming the first violated ceiling; return the RGB image.

    ``Florence2Pipeline.run`` and ``validate_inputs`` both route through this function so their
    acceptance criteria cannot diverge.
    """
    rgb = validate_image(image)
    if task not in TASKS:
        raise ValueError(f"task must be one of TASKS {TASKS}, got {task!r}")
    if task in TASKS_WITH_TEXT:
        if not isinstance(text_input, str) or not text_input.strip():
            raise ValueError(f"task {task} requires a non-empty text_input")
        if len(text_input) > MAX_TEXT_CHARS:
            raise ValueError(f"text_input exceeds MAX_TEXT_CHARS={MAX_TEXT_CHARS}: {len(text_input)}")
    elif text_input is not None:
        raise ValueError(f"task {task} takes no text_input")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    if isinstance(num_beams, bool) or not isinstance(num_beams, int) or num_beams < 1:
        raise TypeError("num_beams must be a positive int")
    return rgb


def validate_inputs(
    image: Image.Image,
    task: str = "<CAPTION>",
    text_input: str | None = None,
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    num_beams: int = NUM_BEAMS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``run`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _check_inputs(image, task, text_input, max_new_tokens, num_beams)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (run takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
            }
        ],
        "task": task,
        "task_requires_text_input": task in TASKS_WITH_TEXT,
        "text_input": text_input,
        "generation": {"max_new_tokens": max_new_tokens, "num_beams": num_beams, "do_sample": False},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any] | Sequence[Mapping[str, Any]],
    reference_text: str | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``result`` is one ``run`` result, or a sequence of them for a multi-capability run. The only
    intrinsic metric in this repository is ``character_error_rate``, and it applies to ``<OCR>``
    when a known transcript is supplied as ``reference_text``; every other capability is
    ``not-measurable`` and the report says what labelled data would make it measurable.
    """
    if not isinstance(result, Mapping):
        subreports = [
            evaluation_report(item, reference_text, sample_kind=sample_kind) for item in result
        ]
        metrics = [
            {**metric, "task": sub["task"]} for sub in subreports for metric in sub["metrics"]
        ]
        return {
            "task": "multi-capability: " + ", ".join(sub["task"] for sub in subreports),
            "score_semantics": _SCORE_SEMANTICS,
            "sample_kind": sample_kind,
            "n_capabilities": len(subreports),
            "metrics": metrics,
            "baselines": [],
            "capabilities": subreports,
            "verdict": "sample-sanity" if metrics else "not-measurable",
            "reason": (
                f"{len(metrics)} capability metric(s) over {len(subreports)} capabilities on one "
                "tutorial sample; sanity evidence, not a benchmark"
                if metrics
                else f"none of the {len(subreports)} demonstrated capabilities has an intrinsic metric here"
            ),
            "needs": "; ".join(dict.fromkeys(sub["needs"] for sub in subreports)),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }
    task = result["task"]
    base = {
        "task": task,
        "score_semantics": _SCORE_SEMANTICS,
        "sample_kind": sample_kind,
        "n_outputs": 1,
        "generation": dict(result.get("generation") or {}),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    family = _TASK_FAMILY.get(task, "region")
    if task == _OCR_TASK and isinstance(reference_text, str) and reference_text.strip():
        reference = reference_text.strip()
        hypothesis = str(result["result"]).strip()
        return {
            **base,
            "metrics": [
                {
                    "id": "character_error_rate",
                    "value": character_error_rate(reference, hypothesis),
                    "reference": reference,
                    "hypothesis": hypothesis,
                    "estimation": "one image against a known transcript, no dispersion estimate",
                }
            ],
            "verdict": "sample-sanity",
            "reason": (
                "one image scored against a transcript the caller already knows; on the synthetic "
                "sample that transcript is text the notebook drew itself, so this is a code-path "
                "check on a rendered font, not an OCR benchmark"
            ),
            "needs": (
                "a labelled OCR corpus from the deployment domain for any generalisable "
                "character-error-rate claim"
            ),
        }
    return {
        **base,
        "metrics": [],
        "verdict": "not-measurable",
        "reason": (
            f"no intrinsic metric exists in this repository for the {family} capability {task}"
            if task != _OCR_TASK
            else "no reference transcript was supplied for the evaluated image"
        ),
        "needs": _NEEDS[family],
    }


def _trainable_names(model: Any) -> list[str]:
    """The last `TRAINABLE_LAYERS` BART decoder layers and the decoder's embedding layer norm; the DaViT vision tower,
    the projector, the BART encoder, the shared embeddings (tied to the output head) and the earlier decoder layers
    stay frozen."""
    return [name for name, _ in model.named_parameters() if name.startswith(_TRAINABLE_PREFIXES)]


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}")
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith(_TRAINABLE_PREFIXES) for n in names):
        raise ValueError(f"artifact tensors must all belong to the last {TRAINABLE_LAYERS} decoder layers or the decoder embedding norm")


@dataclass
class Florence2Pipeline:
    """``_runner(image, prompt, task, max_new_tokens, num_beams)`` -> ``{"text": raw, "parsed": value}``; the optional
    ``_batch_runner(images, max_new_tokens, num_beams)`` -> one ``{"text", "new_tokens"}`` per image for ``<OCR>``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    source: str = "injected"
    dtype: str = "float32"
    _batch_runner: Callable[..., list[dict[str, Any]]] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Florence2Pipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Florence2ForConditionalGeneration, Florence2Processor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        # float32 on every device: the adapter is trained in float32 and overlays without a cast, and CPU,
        # Tesla-class and consumer GPUs then run the same arithmetic.
        dtype = torch.float32
        processor = Florence2Processor.from_pretrained(location, **common)
        model, info = Florence2ForConditionalGeneration.from_pretrained(
            location, dtype=dtype, output_loading_info=True, **common
        )
        bad = {k: v for k, v in info.items() if v}
        if bad:
            raise RuntimeError(f"checkpoint does not match the native Florence-2 architecture: {bad}")
        model = model.eval().to(resolved_device)
        for param in model.parameters():
            param.requires_grad_(False)
        vision_features = model.model.get_image_features

        def chunked_image_features(pixel_values: Any, **kwargs: Any) -> Any:
            """Bound the vision tower's activation memory: `VISION_BATCH_SIZE` images per forward, results concatenated."""
            if pixel_values.shape[0] <= VISION_BATCH_SIZE:
                return vision_features(pixel_values, **kwargs)
            chunks = [vision_features(pixel_values[i : i + VISION_BATCH_SIZE], **kwargs) for i in range(0, pixel_values.shape[0], VISION_BATCH_SIZE)]
            return torch.cat(chunks, dim=0)

        model.model.get_image_features = chunked_image_features
        eos_id = model.config.text_config.eos_token_id
        pad_id = processor.tokenizer.pad_token_id

        def runner(image: Image.Image, prompt: str, task: str, max_new_tokens: int, num_beams: int) -> dict:
            inputs = processor(text=prompt, images=image, return_tensors="pt").to(resolved_device, dtype)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False
                )
            text = processor.batch_decode(generated, skip_special_tokens=False)[0]
            parsed = processor.post_process_generation(text, task=task, image_size=image.size)
            return {"text": text, "parsed": parsed[task]}

        def batch_runner(images: Sequence[Image.Image], max_new_tokens: int, num_beams: int) -> list[dict[str, Any]]:
            inputs = processor(text=[_OCR_TASK] * len(images), images=list(images), return_tensors="pt", padding=True)
            if not bool(inputs["attention_mask"].all()):
                raise RuntimeError("prompts in one batch differ in length; batched generation needs identical prompts")
            inputs = inputs.to(resolved_device, dtype)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False)
            texts = processor.batch_decode(generated, skip_special_tokens=True)
            out = []
            for row, text in zip(generated, texts, strict=True):
                ids = row.tolist()[2:]  # decoder start + <s>
                n_new = len(ids)
                for position, token in enumerate(ids):
                    if token in (eos_id, pad_id):
                        n_new = position + (token == eos_id)
                        break
                out.append({"text": text, "new_tokens": int(n_new)})
            return out

        return cls(runner, resolved_device, source, str(dtype).removeprefix("torch."), batch_runner, model, processor, _weight_digest(root))

    def _validate(
        self, image: Any, task: str, text_input: str | None, max_new_tokens: int, num_beams: int
    ) -> None:
        _check_inputs(image, task, text_input, max_new_tokens, num_beams)

    def run(
        self,
        image: Image.Image,
        task: str = "<CAPTION>",
        text_input: str | None = None,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        num_beams: int = NUM_BEAMS,
    ) -> dict[str, Any]:
        """Run one task prompt on one image; ``result`` is the task-parsed value (text, or boxes + labels)."""
        self._validate(image, task, text_input, max_new_tokens, num_beams)
        prompt = task + (text_input or "")
        raw = self._runner(image.convert("RGB"), prompt, task, max_new_tokens, num_beams)
        if not isinstance(raw, dict) or "text" not in raw or "parsed" not in raw:
            raise RuntimeError("runner must return a dict with 'text' and 'parsed'")
        return {
            "task": task,
            "text_input": text_input,
            "result": raw["parsed"],
            "generated_text": str(raw["text"]),
            "image_size": list(image.size),
            "generation": {"max_new_tokens": max_new_tokens, "num_beams": num_beams, "do_sample": False},
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }


    # ------------------------------------------------------------------------------------------------------
    # Adaptation contract (the <OCR> task on transcribed text lines)
    # ------------------------------------------------------------------------------------------------------

    def transcribe(
        self,
        images: Sequence[Image.Image],
        *,
        max_new_tokens: int = DEFAULT_LINE_MAX_NEW_TOKENS,
        num_beams: int = NUM_BEAMS,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> list[dict[str, Any]]:
        """Run ``<OCR>`` on many images in batches (every image shares the same prompt, so no padding is involved);
        one ``{text, new_tokens, truncated}`` per image, in order. With an injected runner and no batch runner the
        images are recognised one by one through ``run``."""
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        checked = [_check_inputs(image, _OCR_TASK, None, max_new_tokens, num_beams) for image in images]
        out: list[dict[str, Any]] = []
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            if self._batch_runner is not None:
                raw = self._batch_runner(batch, max_new_tokens, num_beams)
            else:
                raw = []
                for image in batch:
                    single = self._runner(image, _OCR_TASK, _OCR_TASK, max_new_tokens, num_beams)
                    raw.append({"text": single["parsed"], "new_tokens": int(single.get("new_tokens", 0))})
            if not isinstance(raw, list) or len(raw) != len(batch) or any(not isinstance(r, dict) or "text" not in r for r in raw):
                raise RuntimeError("batch runner must return one dict with 'text' per image")
            for item in raw:
                new_tokens = int(item.get("new_tokens", 0))
                out.append({"text": normalise_text(str(item["text"])), "new_tokens": new_tokens, "truncated": new_tokens >= max_new_tokens})
            if progress is not None:
                progress(len(out), len(checked))
        return out

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained for adapt/save_artifact/load_artifact")
        return self._model, self._processor

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        max_new_tokens: int = DEFAULT_LINE_MAX_NEW_TOKENS,
        num_beams: int = NUM_BEAMS,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> dict[str, Any]:
        """Transcribe every validated record with ``<OCR>`` and score the hypotheses with ``metrics.ocr_metrics``
        (micro and macro CER / WER, exact match). Works with an injected runner too."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import ocr_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        items = self.transcribe([r["image"] for r in checked], max_new_tokens=max_new_tokens, num_beams=num_beams, batch_size=batch_size, progress=progress)
        hypotheses = [item["text"] for item in items]
        metrics = ocr_metrics(hypotheses, checked)
        metrics.update(
            {
                "hypotheses": hypotheses,
                "truncated": sum(item["truncated"] for item in items),
                "new_tokens": sum(item["new_tokens"] for item in items),
                "max_new_tokens": max_new_tokens,
                "num_beams": num_beams,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None,
        *,
        epochs: int = 6,
        lr: float = 5e-5,
        batch_size: int = 8,
        seed: int = 0,
        max_new_tokens: int = DEFAULT_LINE_MAX_NEW_TOKENS,
        num_beams: int = NUM_BEAMS,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the last `TRAINABLE_LAYERS` BART decoder layers and the decoder's embedding layer norm
        on transcribed lines with the sequence-to-sequence loss over the transcript tokens (``<s>`` … transcript …
        ``</s>``, teacher-forced from the decoder start token) — the model's own objective. The frozen prefix — DaViT
        vision tower, projector and the BART encoder over the ``<OCR>`` prompt — is run once per line under no gradient
        and its encoder output cached, so each step runs only the decoder on those cached states; the loss equals the
        full model's loss exactly. AdamW (no weight decay), gradient clipping at `GRAD_CLIP`, seeded shuffling, no
        scheduler, no augmentation. Epoch 0 records the frozen model's validation metrics; the epoch with the lowest
        validation CER is kept (the final one without a validation split). On any exception the frozen weights are
        restored."""
        model, processor = self._require_model()  # refuse before importing torch
        import torch
        from transformers.modeling_outputs import BaseModelOutput

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        names = _trainable_names(model)
        name_set = set(names)
        device = torch.device(self.device)
        tokenizer = processor.tokenizer
        start_id = int(model.config.text_config.decoder_start_token_id)
        bos_id = int(tokenizer.bos_token_id)
        eos_id = int(tokenizer.eos_token_id)
        pad_id = int(tokenizer.pad_token_id)
        frozen_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        previous_adapter = self.adapter
        cudnn_flags = (torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark)
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False  # repeatable on one device
        history: list[dict[str, Any]] = []
        started = time.perf_counter()

        def _val() -> dict[str, Any] | None:
            if val_checked is None:
                return None
            result = self.evaluate(val_checked, max_new_tokens=max_new_tokens, num_beams=num_beams)
            return {"cer": result["cer"], "wer": result["wer"], "cer_macro": result["cer_macro"], "exact_match": result["exact_match"], "n": result["n"]}

        def _targets(batch: Sequence[Mapping[str, Any]]) -> tuple[Any, Any]:
            seqs = [[start_id, bos_id, *tokenizer(r["text"], add_special_tokens=False)["input_ids"], eos_id] for r in batch]
            length = max(len(seq) for seq in seqs) - 1
            decoder_input = torch.full((len(batch), length), pad_id, dtype=torch.long)
            labels = torch.full((len(batch), length), -100, dtype=torch.long)
            for i, seq in enumerate(seqs):
                decoder_input[i, : len(seq) - 1] = torch.tensor(seq[:-1])
                labels[i, : len(seq) - 1] = torch.tensor(seq[1:])
            return decoder_input, labels

        try:
            # 1. cache the frozen prefix: the encoder output for every training line (identical prompt lengths)
            cache: list[tuple[Any, Any, Any]] = []
            for start in range(0, len(train_checked), CACHE_BATCH_SIZE):
                batch = train_checked[start : start + CACHE_BATCH_SIZE]
                inputs = processor(text=[_OCR_TASK] * len(batch), images=[r["image"] for r in batch], return_tensors="pt", padding=True)
                if not bool(inputs["attention_mask"].all()):
                    raise RuntimeError("prompts in one batch differ in length")
                inputs = inputs.to(device)
                with torch.no_grad():
                    encoded = model.model(**inputs, decoder_input_ids=torch.full((len(batch), 1), start_id, dtype=torch.long, device=device)).encoder_last_hidden_state
                decoder_input, labels = _targets(batch)
                for k in range(len(batch)):
                    cache.append((encoded[k].detach().to("cpu"), decoder_input[k], labels[k]))
                del encoded
            cache_seconds = round(time.perf_counter() - started, 3)
            # 2. train the decoder tail on the cached encoder outputs
            params = []
            for name, param in model.named_parameters():
                if name in name_set:
                    param.requires_grad_(True)
                    params.append(param)
            n_trainable = sum(p.numel() for p in params)
            entry = {"epoch": 0, "train_loss": None, "val": _val(), "note": "frozen model"}
            history.append(entry)
            if progress is not None:
                progress(entry)
            best_epoch, best_score = 0, (history[0]["val"] or {}).get("cer", float("inf"))
            best_state = frozen_state
            optimizer = torch.optim.AdamW(params, lr=float(lr), weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            for epoch in range(1, epochs + 1):
                model.train()
                order = list(range(len(cache)))
                rng.shuffle(order)
                losses = []
                for start in range(0, len(order), batch_size):
                    items = [cache[k] for k in order[start : start + batch_size]]
                    encoded = torch.stack([h for h, _, _ in items]).to(device)
                    length = max(int(d.shape[0]) for _, d, _ in items)
                    decoder_input = torch.full((len(items), length), pad_id, dtype=torch.long)
                    labels = torch.full((len(items), length), -100, dtype=torch.long)
                    for k, (_, d, lab) in enumerate(items):
                        decoder_input[k, : d.shape[0]] = d
                        labels[k, : lab.shape[0]] = lab
                    attention = torch.ones((len(items), encoded.shape[1]), dtype=torch.long, device=device)
                    loss = model(encoder_outputs=BaseModelOutput(last_hidden_state=encoded), attention_mask=attention, decoder_input_ids=decoder_input.to(device), labels=labels.to(device)).loss
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                    optimizer.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": _val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked is None or entry["val"]["cer"] < best_score:
                    best_epoch, best_score = epoch, (entry["val"] or {}).get("cer", float("inf"))
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            model.load_state_dict(best_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
        except BaseException:
            model.load_state_dict(frozen_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
            self.adapter = previous_adapter
            raise
        finally:
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags
        self.adapter = {
            "task": _OCR_TASK,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "first_trainable_layer": _TRAINABLE_FIRST_LAYER,
            "epochs": epochs,
            "batch_size": batch_size,
            "best_epoch": best_epoch,
            "selection": "lowest validation CER" if val_checked is not None else "final epoch (no validation split)",
            "loss": "sequence-to-sequence cross-entropy over the transcript tokens and the end token, teacher-forced from the decoder start token; computed on the cached frozen encoder outputs",
            "lr": float(lr),
            "seed": seed,
            "max_new_tokens": max_new_tokens,
            "num_beams": num_beams,
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "cache_seconds": cache_seconds,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained tensors as safetensors plus a manifest naming the base, the digests and the training
        configuration. Requires a prior `adapt`."""
        model, _processor = self._require_model()  # refuse before importing torch
        import torch
        from safetensors.torch import save_file

        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHTS_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the last decoder layers and the decoder embedding norm."""
        model, _processor = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model)
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Florence2Pipeline:
        """Check the adapter manifest against the base snapshot's recorded weight digest, load the verified base, then
        overlay the adapter (checked again, and the tensor set, before deserialising). A refused manifest never loads
        a model."""
        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        _check_artifact_manifest(manifest, artifact, _weight_digest(root) or "")
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/florence2_vision_language_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level text-recognition measures and two non-neural baselines, in plain Python.

``ocr_metrics`` scores one hypothesis string per record against ``record['text']``: the character error rate and the
word error rate as **micro** averages (total Levenshtein edits over total reference characters or words — the
corpus CER/WER of the handwriting-recognition literature) and as **macro** averages (mean per-line rate), the exact-
match rate, and the counts behind them. Neither rate is capped: a hypothesis longer than its reference can push a
rate above 1.0, which is the signal that the model is generating text the image does not carry. The baselines
answer without looking at the image — the empty string, or one constant training transcript — and are scored by
the same function.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import edit_distance, normalise_text` removed — names are kernel globals defined by the carried modules

METRIC_DEFINITIONS = {
    "cer": "character error rate, micro: total character edits (insertions + deletions + substitutions) over total reference characters; 0 is perfect, values above 1 mean over-generation",
    "wer": "word error rate, micro: total word edits over total reference words after lower-casing and whitespace tokenisation; punctuation kept",
    "cer_macro": "mean of the per-line character error rates (each line weighted equally regardless of length)",
    "wer_macro": "mean of the per-line word error rates",
    "exact_match": "fraction of lines whose hypothesis equals the reference after whitespace normalisation",
}
MEDOID_POOL = 120  # training transcripts considered when choosing the constant baseline (quadratic cost)


def _words(text: str) -> list[str]:
    return text.lower().split()


def ocr_metrics(hypotheses: Sequence[str], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Score one hypothesis per record against `record['text']`; per-line rows plus the corpus rates. Raises when
    the lengths differ or nothing is scored."""
    if len(hypotheses) != len(records) or not records:
        raise ValueError("hypotheses and records must be non-empty and the same length")
    rows = []
    char_edits = word_edits = ref_chars = ref_words = hyp_chars = exact = 0
    for hypothesis, record in zip(hypotheses, records, strict=True):
        reference = normalise_text(record["text"])
        hyp = normalise_text(str(hypothesis))
        if not reference:
            raise ValueError(f"record {record.get('id')!r} has an empty reference")
        ce = edit_distance(reference, hyp)
        we = edit_distance(_words(reference), _words(hyp))
        n_words = len(_words(reference))
        rows.append({"id": record.get("id"), "ref_chars": len(reference), "hyp_chars": len(hyp), "char_edits": ce, "cer": ce / len(reference), "word_edits": we, "wer": we / n_words, "exact": hyp == reference})
        char_edits += ce
        word_edits += we
        ref_chars += len(reference)
        ref_words += n_words
        hyp_chars += len(hyp)
        exact += hyp == reference
    n = len(rows)
    return {
        "n": n,
        "cer": char_edits / ref_chars,
        "wer": word_edits / ref_words,
        "cer_macro": sum(r["cer"] for r in rows) / n,
        "wer_macro": sum(r["wer"] for r in rows) / n,
        "exact_match": exact / n,
        "char_edits": char_edits,
        "ref_chars": ref_chars,
        "hyp_chars": hyp_chars,
        "word_edits": word_edits,
        "ref_words": ref_words,
        "rows": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def empty_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Predict the empty string for every line: CER and WER are exactly 1.0 by construction (every reference
    character is a deletion). The floor any recogniser must beat to be doing better than silence."""
    result = ocr_metrics([""] * len(records), records)
    result["baseline"] = "empty string for every line"
    return result


def medoid_transcript(train: Sequence[Mapping[str, Any]], *, pool: int = MEDOID_POOL) -> str:
    """The training transcript (among the first `pool`) with the smallest summed character error rate to the others:
    the single constant string that best matches the corpus on average."""
    texts = [normalise_text(r["text"]) for r in train[:pool]]
    texts = [t for t in texts if t]
    if not texts:
        raise ValueError("no non-empty training transcripts")
    return min(texts, key=lambda candidate: sum(edit_distance(other, candidate) / len(other) for other in texts))


def constant_baseline(train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]], *, pool: int = MEDOID_POOL) -> dict[str, Any]:
    """Predict one constant training transcript (the medoid) for every line: what corpus statistics alone buy
    without reading the image — usually a CER near 1.0, since edits are dominated by substitutions."""
    text = medoid_transcript(train, pool=pool)
    result = ocr_metrics([text] * len(records), records)
    result["baseline"] = f"constant training transcript {text!r}"
    result["transcript"] = text
    return result

**Module 3/3:** `src/florence2_vision_language_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled text-line datasets for the adaptation contract: the digest-pinned Belfort sample, the record contract
and its structural validation, image-disjoint splitting, and the BYOD loader.

A record is ``{id, image, text}`` where ``image`` is a PIL image of one text line (or any region whose transcript is
known; sides within the pipeline's ceilings) and ``text`` its transcript — the string the model is asked to
generate for that image. Whitespace runs in ``text`` are collapsed; case and punctuation are kept.

The default sample is drawn from the Belfort-line dataset (Teklia; the minutes of the Belfort municipal council,
19th–20th century French handwriting transcribed by a crowdsourcing campaign; Tarride et al. 2023, **MIT**) as
converted to parquet by the Hugging Face Hub at an immutable revision: the first ``CORPUS_ROW_GROUPS`` row groups of
the test shard are read with HTTPS range requests (about 5.5 MB each; the shard's declared size is checked first and
every row group's decoded content is refused unless its SHA-256 matches the pin). Every line image is 128 px tall;
widths run from about 145 to 9,000 px. The domain gap to the model's printed-text training distribution is the point
of the sample: the frozen model reads almost none of it.
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MODEL_ID, normalise_text, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Belfort-line (test split), first eight parquet row groups"
CORPUS_REPO = "Teklia/Belfort-line"
CORPUS_REVISION = "c4a74bbd39f2df314752e7e6026649a39d365cbb"  # refs/convert/parquet commit on the Hub
CORPUS_FILE = "default/test/0000.parquet"
CORPUS_BYTES = 210_579_166
CORPUS_ROWS = 3_819
CORPUS_ROW_GROUPS = 8  # of 39; 100 lines each
CORPUS_LICENSE = "MIT (Teklia; Belfort municipal council minutes, Zenodo record 8041668; Tarride et al. 2023, https://doi.org/10.1145/3604951.3605517)"
CORPUS_LANGUAGE = "fr"
CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"
# SHA-256 over the concatenated image bytes + UTF-8 transcript of each row group, in row order, and that byte total.
ROW_GROUP_PINS: dict[int, tuple[str, int]] = {
    0: ("1dc3141e4809ea628b17c3ca7b81d64e6ca92bce18dd5765ecd618bfc7867954", 5_481_145),
    1: ("c9d3b52013933c803f4886edbce68da0ae483347a4a6ff3e0f5ced1db4a7e653", 5_465_901),
    2: ("6e0578a90a07a9e25e65b765881d3fa33d6a797624425e01026980d7287f0bf6", 5_379_166),
    3: ("00cdfb7aabe924f31b9f1bb1ba4849040051e5619567b68bf99fdcbcab15131a", 5_821_303),
    4: ("fc063442fb20e7a60c2533ab44dcc69a22ad59f5ce24921fe6af5f53ceab7e1a", 5_163_559),
    5: ("2d7e29331bd4e93e0c8a1caa9a83b8f1ede9b17af6dae9377b83f56c56f05689", 4_713_140),
    6: ("49423d91780cb184c4b0069630e85acccb314a113256ced9692a5138c4782ef1", 5_069_794),
    7: ("3a010831456f185399579b4ecf9d46222f95368c3cdbbc2ff103a258b9c16e2f", 5_309_891),
}
DEFAULT_CACHE_DIR = Path("weights") / "belfort"

SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 600, "validation": 60, "test": 140}  # of the 800 lines the eight row groups hold
SAMPLE_DIGEST = "b7e1dd684691a0eedb63a609311f4964e7732e5c1a8d254fe4e1293a8cd0964d"  # dataset_digest over the three default splits together; tests pin it
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MIN_TEXT_CHARS = 1
MAX_TEXT_CHARS = 512
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to read a
    parquet footer and a few row groups without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}", "User-Agent": "florence2-vision-language-pipeline"})
        with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _declared_size(url: str) -> int:
    request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "florence2-vision-language-pipeline"})
    with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 (pinned https URL)
        length = response.headers.get("Content-Length")
    if length is None:
        raise ValueError(f"{url}: no Content-Length in the HEAD response")
    return int(length)


def _group_digest(rows: Sequence[Mapping[str, Any]]) -> tuple[str, int]:
    digest, total = hashlib.sha256(), 0
    for row in rows:
        data = row["image"]["bytes"]
        text = str(row["text"]).encode("utf-8")
        digest.update(data)
        digest.update(text)
        total += len(data) + len(text)
    return digest.hexdigest(), total


def fetch_corpus(
    *, cache_dir: str | Path | None = None, groups: Sequence[int] | None = None, opener: Any = None
) -> dict[int, list[dict[str, Any]]]:
    """Return the pinned row groups as lists of `{image, text}` (JPEG bytes, transcript), from the cache (one parquet
    file per row group) or the Hub (footer + the row groups it needs, over range requests). Every row group's decoded
    content is refused unless its SHA-256 and byte total match `ROW_GROUP_PINS`; a fresh fetch also checks the shard's
    declared size and row count."""
    import pyarrow.parquet as pq

    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    wanted = list(groups) if groups is not None else sorted(ROW_GROUP_PINS)
    out: dict[int, list[dict[str, Any]]] = {}
    reader = None
    for group in wanted:
        if group not in ROW_GROUP_PINS:
            raise ValueError(f"row group {group} has no pin; pinned groups are {sorted(ROW_GROUP_PINS)}")
        local = cache / f"test-rg{group}.parquet"
        rows: list[dict[str, Any]] | None = None
        if local.is_file():
            rows = pq.read_table(local).to_pylist()
            if _group_digest(rows) != ROW_GROUP_PINS[group]:
                rows = None  # stale or corrupt cache: refetch
        if rows is None:
            if reader is None:
                if opener is not None:
                    reader = pq.ParquetFile(opener(CORPUS_URL))
                else:
                    declared = _declared_size(CORPUS_URL)
                    if declared != CORPUS_BYTES:
                        raise ValueError(f"{CORPUS_FILE}: declared size {declared} != pinned {CORPUS_BYTES}")
                    reader = pq.ParquetFile(_HttpRangeFile(CORPUS_URL, CORPUS_BYTES))
                if reader.metadata.num_rows != CORPUS_ROWS:
                    raise ValueError(f"{CORPUS_FILE}: {reader.metadata.num_rows} rows, pinned {CORPUS_ROWS}")
            table = reader.read_row_group(group, columns=["image", "text"])
            rows = table.to_pylist()
            digest, total = _group_digest(rows)
            if (digest, total) != ROW_GROUP_PINS[group]:
                raise ValueError(f"{CORPUS_FILE} row group {group}: sha256 {digest} / {total} bytes != pinned {ROW_GROUP_PINS[group]}")
            pq.write_table(table, local)
        out[group] = [{"image": r["image"]["bytes"], "text": str(r["text"])} for r in rows]
    return out


def read_corpus(groups: Mapping[int, Sequence[Mapping[str, Any]]]) -> list[dict[str, Any]]:
    """Decode the verified row groups into records (one per line; lines with an empty transcript are skipped)."""
    out = []
    for group in sorted(groups):
        for index, row in enumerate(groups[group]):
            text = normalise_text(row["text"])
            if not text:
                continue
            image = Image.open(io.BytesIO(row["image"]))
            image.load()
            out.append({"id": f"belfort-test-{group * 100 + index}", "image": image.convert("RGB"), "text": text, "source_row_group": group})
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """Seeded line-level draw: shuffle the records and cut `sizes` (train / validation / test) in order."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    pool = [dict(r) for r in records]
    random.Random(seed).shuffle(pool)
    needed = sum(sizes.values())
    if len(pool) < needed:
        raise ValueError(f"only {len(pool)} records available, need {needed}")
    out, cursor = {}, 0
    for name, count in sizes.items():
        out[name] = pool[cursor : cursor + count]
        cursor += count
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)


# ---------------------------------------------------------------------------------------------------------
# Record contract
# ---------------------------------------------------------------------------------------------------------


def _open(image: Any, where: str) -> Image.Image:
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{where}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{where}: must be a PIL.Image.Image or a file path")
    return image


def _check_record(record: Any, index: int) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/image/text")
    for key in ("id", "image", "text"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    try:
        image = validate_image(_open(record["image"], f"{where}.image"))
    except TypeError as exc:
        raise ValueError(f"{where}: {exc}") from exc
    except ValueError as exc:
        raise ValueError(f"{where}: {exc}") from exc
    if not isinstance(record["text"], str):
        raise ValueError(f"{where}: text must be a str")
    text = normalise_text(record["text"])
    if not MIN_TEXT_CHARS <= len(text) <= MAX_TEXT_CHARS:
        raise ValueError(f"{where}: text has {len(text)} characters after whitespace normalisation; {MIN_TEXT_CHARS}..{MAX_TEXT_CHARS} are required")
    item = {"id": rid, "image": image, "text": text}
    if "source_row_group" in record:
        item["source_row_group"] = record["source_row_group"]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a text-line dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, text} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked, ids = [], set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    chars = [len(r["text"]) for r in checked]
    words = [len(r["text"].split()) for r in checked]
    widths = [r["image"].width for r in checked]
    heights = [r["image"].height for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "text_chars": {"min": min(chars), "max": max(chars), "total": sum(chars)},
        "text_words": {"total": sum(words)},
        "image_width": {"min": min(widths), "max": max(widths)},
        "image_height": {"min": min(heights), "max": max(heights)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size-prefixed) — the identity a split is made disjoint on."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over (id, image digest, normalised text)."""
    parts = sorted(f"{r['id']}:{image_digest(r['image'])}:{normalise_text(r['text'])}" for r in records)
    return _sha256_bytes("\n".join(parts).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n = len(unique)
    n_test = max(1, round(n * test_fraction))
    n_val = round(n * val_fraction)
    if n - n_test - n_val < 1:
        raise ValueError(f"{n} distinct images are too few to split into train/validation/test")
    return {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a directory or zip holding line images and a `transcripts.csv` with the columns `file` and `text`
    (and optionally `id`); every image file must have a transcript row and every row an image."""
    source = Path(path)
    members: dict[str, bytes] = {}
    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file():
                members[file.name] = file.read_bytes()
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                if not info.is_dir():
                    members[Path(info.filename).name] = archive.read(info)  # flattened; no extractall
    else:
        raise ValueError(f"{source} is neither a directory nor a zip file")
    if "transcripts.csv" not in members:
        raise ValueError("BYOD data must include transcripts.csv with the columns file and text")
    rows = list(csv.DictReader(io.StringIO(members["transcripts.csv"].decode("utf-8-sig"))))
    if not rows or "file" not in rows[0] or "text" not in rows[0]:
        raise ValueError("transcripts.csv must have the columns file and text")
    out = []
    for row in rows:
        name = Path(str(row.get("file", "")).strip()).name
        if name not in members:
            raise ValueError(f"transcripts.csv names a missing image: {name}")
        try:
            image = Image.open(io.BytesIO(members[name]))
            image.load()
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"BYOD file is not a decodable image: {name}") from exc
        rid = str(row.get("id", "") or "").strip()
        out.append({"id": rid or re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "image": image.convert("RGB"), "text": str(row.get("text", ""))})
    listed = {Path(str(r.get("file", "")).strip()).name for r in rows}
    unlisted = [n for n in members if n != "transcripts.csv" and n not in listed]
    if unlisted:
        raise ValueError(f"{len(unlisted)} image file(s) have no transcripts.csv row, e.g. {unlisted[0]}")
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A summary table (id, image size, transcript length, transcript, provenance) in the BYOD `transcripts.csv`
    column layout plus extras (`file` names the id; the images themselves are not written)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "file", "width", "height", "chars", "words", "text", "source_row_group"])
        for r in records:
            text = normalise_text(r["text"])
            writer.writerow([r["id"], f"{r['id']}.jpg", r["image"].width, r["image"].height, len(text), len(text.split()), text, r.get("source_row_group", "")])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `12`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `4271c66b88cd…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Florence2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "florence-2-large-community",
  "modelId": "florence-community/Florence-2-large",
  "revision": "4271c66b88cdbc05735372ec13b2360108de5317",
  "files": [
    {
      "path": "README.md",
      "bytes": 13445,
      "sha256": "5662d3853b245397062aa0b1853958f23305e0b9518071293c5154c7eeb415d2"
    },
    {
      "path": "added_tokens.json",
      "bytes": 22430,
      "sha256": "1d75deda84dfa81fb6c09301f3fed00f9695059568bbb1403a6bf299cd84fc37"
    },
    {
      "path": "config.json",
      "bytes": 2396,
      "sha256": "8412483f687f2f71587328a38c6fa70a68d9488f28e90607be3c182641f60f2c"
    },
    {
      "path": "generation_config.json",
      "bytes": 292,
      "sha256": "0251459c49cc358ac033b5d4b8569e61ac22bde5d76be404b97a44cfb33fb12e"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1553541016,
      "sha256": "7715423d6549bf1e71188bdd84f4ac960cc0597886af24a5ef7b66f128660685"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 603,
      "sha256": "1396ec5a0a7adfe1c04fb777b09e8ba753be6dbb5868212ab3c3ef39d91fe031"
    },
    {
      "path": "processor_config.json",
      "bytes": 2264,
      "sha256": "cd0e3bf41a39b1276503fbd273bc03b9afc70d7a15ea681a92a1b4b77f858ee6"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 146627,
      "sha256": "72ff172dc769bc1551b1b4211628ce3271643bc60379e4da45d85a9be9332c39"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3748144,
      "sha256": "3ad7001f773409abe6bba33eac92662611a73d72f459bda2f00d2a221dd31ce4"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 197922,
      "sha256": "cb5f80bd9afa767bb1bb798ee5f0a79eae45239b8e72506166b4218175a16723"
    },
    {
      "path": "vocab.json",
      "bytes": 798293,
      "sha256": "ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e"
    }
  ],
  "totalBytes": 1558929750
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Florence2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Belfort lines, the transcripts and the split

`fetch_corpus` returns the eight pinned row groups from the cache under `weights/belfort/` or the Hub at the pinned parquet-conversion revision — `pyarrow` reads the shard's footer and exactly those row groups over HTTPS range requests; every cached file is re-hashed and every fetched row group refused on any SHA-256 or byte-total mismatch — and `read_corpus` turns each row into a record: the line image (128 px tall, 145 to 8,956 px wide) and its crowdsourced transcript with whitespace runs collapsed. `build_sample_dataset` draws a seeded line-level split (600 / 60 / 140). `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no image (by decoded-pixel digest) is shared, and the training split's summary table is written to `outputs/florence2_vision_language_train.csv`.

Look for: 800 lines and 33,117 reference characters, three digests, and four refusal probes — a duplicate id, an empty transcript, an image above the side ceiling, and a dataset too small to use — each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    t0 = time.perf_counter()
    corpus_groups = fetch_corpus(cache_dir='weights/belfort')
    corpus = read_corpus(corpus_groups)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} @ {CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'row_groups': len(corpus_groups), 'lines': sum(len(v) for v in corpus_groups.values()), 'bytes': sum(len(r['image']) + len(r['text'].encode('utf-8')) for v in corpus_groups.values() for r in v), 'seconds': round(time.perf_counter() - t0, 1)}
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(train_records, 'outputs/florence2_vision_language_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'chars': manifest['text_chars'], 'words': manifest['text_words']['total'], 'width': manifest['image_width'], 'height': manifest['image_height'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
example['image'].save('outputs/florence2_vision_language_example_line.png')
print({'example': {'id': example['id'], 'image': list(example['image'].size), 'text': example['text']}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty transcript': [{**train_records[0], 'text': '   '}, *train_records[1:8]],
    'image above the side ceiling': [{**train_records[0], 'image': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8))}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Run three capabilities on a synthetic drawing through the inference contract

The inference contract is exercised as the multi-capability tutorial exercised it: a 512 × 512 white canvas drawn in code — a filled red square, a filled blue circle and the text `DIMER 2026` in Pillow's bundled font — whose drawn text is the **known OCR reference**; a different image family from the handwritten lines, and a drawing the adapted model will see again in Section 9. `validate_inputs` applies exactly the checks `run` applies (image sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` and at most `MAX_IMAGE_PIXELS`, a task token inside `TASKS`, a `text_input` only for the tasks that take one, `max_new_tokens` in 1..`MAX_NEW_TOKENS`, a positive `num_beams`) and one combined manifest records the three requests; an unsupported task token is validated too and its rejection recorded as a finding. `run` returns the task-parsed `result` — a caption string, `{bboxes, labels}` in input pixels with **no confidence scores**, an OCR string — the raw `generated_text` and the generation settings; structural checks assert each output's contract. The per-image `evaluation_report` over the three results scores OCR against the drawn text (`sample-sanity`) and reports the caption and the detections `not-measurable` — plumbing evidence, not a measurement; whether the model is *good at handwriting* is what Section 6 measures on 140 lines. The multi-capability card recorded the square labelled `flag` and an exact OCR of the drawn text.

In [ ]:
drawing = Image.new('RGB', (512, 512), (255, 255, 255))
draw = ImageDraw.Draw(drawing)
drawn_square = (64, 64, 224, 224)
draw.rectangle(drawn_square, fill=(220, 30, 30))
draw.ellipse((300, 96, 460, 256), fill=(30, 60, 220))
drawing_reference = 'DIMER 2026'
draw.text((96, 360), drawing_reference, fill=(0, 0, 0), font=ImageFont.load_default(size=48))
drawing_name = 'synthetic_shapes_text_512'
drawing_sha256 = hashlib.sha256(np.asarray(drawing).tobytes()).hexdigest()
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_IMAGE_PIXELS': MAX_IMAGE_PIXELS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DEFAULT_LINE_MAX_NEW_TOKENS': DEFAULT_LINE_MAX_NEW_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'NUM_BEAMS': NUM_BEAMS, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'device': pipe.device, 'dtype': pipe.dtype}})
print({'TASKS_WITHOUT_TEXT': TASKS_WITHOUT_TEXT, 'TASKS_WITH_TEXT': TASKS_WITH_TEXT})
CAPABILITIES = {
    '<CAPTION>': {'input': 'image only', 'output': 'result: str (one short caption); no score', 'max_new_tokens': 64},
    '<OD>': {'input': 'image only', 'output': "result: {'bboxes': [[x1, y1, x2, y2], ...] in input pixels, 'labels': [str, ...]}; no per-box score", 'max_new_tokens': DEFAULT_MAX_NEW_TOKENS},
    '<OCR>': {'input': 'image only', 'output': 'result: str (transcribed text, reading order chosen by the model); no score', 'max_new_tokens': DEFAULT_LINE_MAX_NEW_TOKENS},
}
manifests = {task: validate_inputs(drawing, task, max_new_tokens=contract['max_new_tokens'], num_beams=NUM_BEAMS, names=[drawing_name]) for task, contract in CAPABILITIES.items()}
input_manifest = {**manifests['<CAPTION>'], 'task': 'multi-capability: ' + ', '.join(CAPABILITIES), 'tasks': list(CAPABILITIES), 'findings': [], 'capabilities': manifests}
try:
    validate_inputs(drawing, '<REFERRING_EXPRESSION_SEGMENTATION>')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'unsupported-task-probe', 'task': '<REFERRING_EXPRESSION_SEGMENTATION>', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/florence2_vision_language_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'drawing': drawing_name, 'sha256': drawing_sha256[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def run_capabilities(pipeline, label):
    results, timings = {}, {}
    for task, contract in CAPABILITIES.items():
        started = time.perf_counter()
        results[task] = pipeline.run(drawing, task, max_new_tokens=contract['max_new_tokens'], num_beams=NUM_BEAMS)
        timings[task] = round(time.perf_counter() - started, 3)
    caption, detections, ocr_text = results['<CAPTION>']['result'], results['<OD>']['result'], results['<OCR>']['result']
    checks = {
        'caption_is_text': isinstance(caption, str),
        'od_boxes_and_labels_aligned': isinstance(detections, dict) and len(detections.get('bboxes', [])) == len(detections.get('labels', [])),
        'od_boxes_inside_image': all(0 <= x1 <= x2 <= drawing.width and 0 <= y1 <= y2 <= drawing.height for x1, y1, x2, y2 in detections.get('bboxes', [])),
        'ocr_is_text': isinstance(ocr_text, str),
        'deterministic_settings': all(r['generation']['do_sample'] is False and r['generation']['num_beams'] == NUM_BEAMS for r in results.values()),
        'identity_reported': all(r['model_id'] == MODEL_ID and r['model_revision'] == MODEL_REVISION for r in results.values()),
    }
    if not all(checks.values()):
        raise RuntimeError(f'capability output failed a sanity check: {checks}')
    report = evaluation_report(list(results.values()), drawing_reference, sample_kind='synthetic (drawn in this notebook)')
    preview = drawing.copy()
    marker = ImageDraw.Draw(preview)
    for (x1, y1, x2, y2), name in zip(detections.get('bboxes', []), detections.get('labels', [])):
        marker.rectangle((x1, y1, x2, y2), outline=(0, 160, 0), width=3)
        marker.text((x1 + 4, y1 + 4), name, fill=(0, 160, 0))
    preview.save(f'outputs/florence2_vision_language_preview_{label}.png')
    print({label: {'seconds': timings, 'checks': checks, 'caption': caption, 'detections': list(zip(detections.get('labels', []), [[round(v) for v in b] for b in detections.get('bboxes', [])])), 'ocr': ocr_text, 'ocr_cer': {m['id']: round(m['value'], 4) for m in report['metrics']}, 'verdict': report['verdict']}})
    return results, timings, checks, report


frozen_results, frozen_timings, frozen_checks, frozen_report = run_capabilities(pipe, 'frozen')

## 6. Baselines and the frozen model on the test lines

Two non-adapted baselines frame the adaptation, each scored by `ocr_metrics` (carried in `metrics.py`): the **character error rate** and **word error rate** as micro averages — total Levenshtein edits over total reference characters or words, the corpus CER/WER of the handwriting-recognition literature — beside the macro (per-line mean) rates and the exact-match rate. Neither rate is capped: a hypothesis longer than its reference pushes the rate **above 1.0**, the signal that the model is generating text the line does not carry. The **empty-string** baseline predicts nothing and scores CER 1.0 exactly (every reference character is a deletion) — the floor any recogniser must beat to do better than silence. The **constant-transcript** baseline predicts one training transcript — the medoid, the line closest on average to the others — for every test line: what corpus statistics buy without reading the image. The **frozen model** is scored by `pipe.evaluate`, which runs `<OCR>` in batches of `EVAL_BATCH_SIZE` with `NUM_BEAMS` beams under a `LINE_MAX_NEW_TOKENS` budget and returns the hypotheses with the rates. Expect the frozen model **at the empty baseline**: the build record measured @P:FROZEN_CER@ — it emits a dash or nothing for cursive it cannot read (hypotheses @P:FROZEN_HYP_RATIO@ times the reference length); read four of them under the references.

In [ ]:
METRICS = ('cer', 'wer', 'cer_macro', 'exact_match')
LINE_MAX_NEW_TOKENS = 128  # @param {type:"integer"}

baseline_empty = empty_baseline(test_records)
baseline_constant = constant_baseline(train_records, test_records)
print({'empty_baseline': {k: round(baseline_empty[k], 3) for k in METRICS}, 'n': baseline_empty['n'], 'note': baseline_empty['baseline']})
print({'constant_baseline': {k: round(baseline_constant[k], 3) for k in METRICS}, 'note': baseline_constant['baseline']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, batch_size=EVAL_BATCH_SIZE)
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'n': frozen_test['n'], 'ref_chars': frozen_test['ref_chars'], 'hyp_chars': frozen_test['hyp_chars'], 'truncated': frozen_test['truncated'], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
for record, hypothesis in zip(test_records[:4], frozen_test['hypotheses'][:4], strict=True):
    print({'id': record['id'], 'reference': record['text'], 'frozen': hypothesis[:120]})

## 7. Bounded fine-tuning of the last decoder layers

`pipe.adapt` trains only the last four of the 12 BART decoder layers and the decoder's embedding layer norm — 67,188,736 of 776,505,344 parameters — while the DaViT vision tower, the projector, the BART encoder, the shared embeddings (tied to the output head) and the first eight decoder layers stay frozen. Each training line is the `<OCR>` prompt with its 577 visual tokens on the encoder side and, on the decoder side, the transcript's tokens between the start and end tokens; the loss is the **sequence-to-sequence cross-entropy** over those target tokens, teacher-forced — the checkpoint's own training objective. Because everything before the decoder is frozen, the encoder output for every training line is computed once under no gradient and cached (the **frozen encoder cache**, 587 × 1024 numbers per line), and each step runs only the decoder on those cached states — the loss equals the full model's loss exactly, at a fraction of the cost. AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, seeded shuffling, no scheduler, no augmentation. Epoch 0 records the frozen model's validation rates; every epoch is scored on the 60 validation lines with the same beam search, and the epoch with the **lowest validation CER** is kept.

Watch the validation CER fall from @P:VAL_CER_0@ to @P:VAL_CER_BEST@ (epoch @P:BEST_EPOCH@ in the build record) while the loss drops from about @P:LOSS_1@ to @P:LOSS_LAST@ — and then watch the validation curve flatten near 0.80 while the training loss keeps falling. Four decoder layers move the model from silence to something that is not yet reading (@P:ADAPTED_READ@): the frozen encoder's representation of a 128-px cursive line squashed into 768 × 768 is the bottleneck, not the optimiser — the build record's faster rates (1e-4 and 2e-4 for eight epochs) reached the same plateau (test CER 0.798 and 0.807). The sibling GOT-OCR 2.0 row reached 0.759 on the same split with the same design; the two models are compared in the card, not here.

In [ ]:
EPOCHS = 6  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in METRICS})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'task': adapt_result['task'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'first_trainable_layer': adapt_result['first_trainable_layer'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'loss': adapt_result['loss'], 'cache_seconds': adapt_result['cache_seconds'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test lines were never used for training or epoch selection, and no image appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6 and the four systems are put side by side. Read it in this order: **CER** first (the measure the epoch was selected on — the build record measured @P:FROZEN_CER@ → **@P:ADAPTED_CER@**, past both baselines), then **WER** (@P:FROZEN_WER@ → @P:ADAPTED_WER@: whole words, not just characters), then the hypothesis length (from @P:FROZEN_HYP_RATIO@ times the reference length to @P:ADAPTED_HYP_RATIO@: the adapted model now writes lines of the right length), then the exact-match rate (@P:ADAPTED_EXACT@ of the 140 lines read perfectly). The cell asserts the adapted CER is below the frozen one and below the empty baseline's 1.0. One hundred and forty lines from one seeded split give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on one French council's minutes says nothing about other hands, other languages or other scripts until you measure them.

In [ ]:
adapted_test = pipe.evaluate(test_records, max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, batch_size=EVAL_BATCH_SIZE)
adapted_val = pipe.evaluate(val_records, max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, batch_size=EVAL_BATCH_SIZE)
comparison = {metric: {'empty': round(baseline_empty[metric], 3), 'constant': round(baseline_constant[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
comparison['hypothesis_length'] = {'ref_chars': adapted_test['ref_chars'], 'frozen_hyp_chars': frozen_test['hyp_chars'], 'adapted_hyp_chars': adapted_test['hyp_chars'], 'frozen_truncated': frozen_test['truncated'], 'adapted_truncated': adapted_test['truncated']}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'task': '<OCR>',
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'empty': {k: v for k, v in baseline_empty.items() if k != 'rows'}, 'constant': {k: v for k, v in baseline_constant.items() if k != 'rows'}},
    'frozen_test': {k: v for k, v in frozen_test.items() if k != 'rows'},
    'validation_metrics': {k: v for k, v in adapted_val.items() if k != 'rows'},
    'test_metrics': {k: v for k, v in adapted_test.items() if k != 'rows'},
    'per_line': [{**frozen_row, 'frozen_hypothesis': frozen_hyp, 'adapted_cer': adapted_row['cer'], 'adapted_hypothesis': adapted_hyp} for frozen_row, frozen_hyp, adapted_row, adapted_hyp in zip(frozen_test['rows'], frozen_test['hypotheses'], adapted_test['rows'], adapted_test['hypotheses'], strict=True)],
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/florence2_vision_language_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['cer'] < frozen_test['cer']
assert adapted_test['cer'] < baseline_empty['cer']
print({'report': 'outputs/florence2_vision_language_evaluation_report.json', 'adapted_beats_both_baselines': adapted_test['cer'] < min(baseline_empty['cer'], baseline_constant['cer'])})

## 9. Look at the lines, re-run the three capabilities, export the adapter and reload it

Six held-out lines are written as panels (`outputs/florence2_vision_language_examples/`: the line image with the reference, the frozen transcript and the adapted transcript beneath it) so the numbers can be checked by eye: the adapted rows should read the cursive the frozen rows left blank. The drawing from Section 5 is then run again through all three capabilities by the adapted model — the decoder that was tuned serves every task token, so this is a small look at what the adaptation did *outside* its task and its corpus: the build record measured @P:DRAWING_AFTER@ — one drawing of evidence, not a measurement.

`pipe.save_artifact` writes the trained tensors — the four decoder layers and the embedding norm, about 269 MB in float32 — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the task, the training configuration and the epoch history (OUT8). `Florence2Pipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the last four decoder layers and the embedding norm, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical transcripts on eight test lines (VER4).

In [ ]:
import shutil

examples_dir = Path('outputs/florence2_vision_language_examples')
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir(parents=True)
caption_font = ImageFont.load_default(size=18)
for record, frozen_hyp, adapted_hyp in zip(test_records[:6], frozen_test['hypotheses'][:6], adapted_test['hypotheses'][:6], strict=True):
    line = record['image']
    width = min(1400, line.width)
    line = line.resize((width, max(1, round(line.height * width / record['image'].width))))
    sheet = Image.new('RGB', (max(width, 1400), line.height + 96), (255, 255, 255))
    sheet.paste(line, (0, 0))
    marker = ImageDraw.Draw(sheet)
    for i, (tag, text) in enumerate((('REF', record['text']), ('FROZEN', frozen_hyp), ('ADAPTED', adapted_hyp))):
        marker.text((8, line.height + 6 + i * 28), f'{tag}: {text[:140]}', fill=(20, 20, 20) if tag != 'FROZEN' else (150, 40, 40), font=caption_font)
    sheet.save(examples_dir / f"{record['id']}.png")
print({'examples': sorted(p.name for p in examples_dir.iterdir()), 'rows': ['reference', 'frozen transcript', 'adapted transcript']})

adapted_results, adapted_timings, adapted_checks, adapted_report = run_capabilities(pipe, 'adapted')

artifact_dir = Path('outputs/florence2_vision_language_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'florence2_vision_language', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'task': artifact_manifest['adapter']['task'], 'best_epoch': artifact_manifest['adapter']['best_epoch']})

reloaded = Florence2Pipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [item['text'] for item in pipe.transcribe([r['image'] for r in test_records[:8]], max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)]
after = [item['text'] for item in reloaded.transcribe([r['image'] for r in test_records[:8]], max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)]
parity = {'identical_lines': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_lines'] == parity['of']

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'file': CORPUS_FILE, 'license': CORPUS_LICENSE, 'language': CORPUS_LANGUAGE, 'row_groups': sorted(ROW_GROUP_PINS), 'shard_bytes': CORPUS_BYTES},
    'inference_contract': {'input_manifest': input_manifest, 'drawing': {'name': drawing_name, 'sha256': drawing_sha256, 'ocr_reference': drawing_reference, 'drawn_square': drawn_square}, 'tasks_exposed': list(TASKS), 'frozen': {'capabilities': {task: {'result': r['result'], 'generated_text': r['generated_text'], 'generation': r['generation'], 'seconds': frozen_timings[task]} for task, r in frozen_results.items()}, 'checks': frozen_checks, 'report': frozen_report}, 'adapted': {'capabilities': {task: {'result': r['result'], 'generated_text': r['generated_text'], 'generation': r['generation'], 'seconds': adapted_timings[task]} for task, r in adapted_results.items()}, 'checks': adapted_checks, 'report': adapted_report}, 'output_files': ['outputs/florence2_vision_language_preview_frozen.png', 'outputs/florence2_vision_language_preview_adapted.png']},
    'comparison': comparison,
    'examples': 'outputs/florence2_vision_language_examples',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pillow': PIL.__version__, 'device': pipe.device, 'source': pipe.source, 'dtype': pipe.dtype},
}
with open('outputs/florence2_vision_language_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A prompt-driven document model reads the print it was trained on, and nineteenth-century cursive French is not among it: the frozen `<OCR>` scores a character error rate of @P:FROZEN_CER@ on the Belfort lines — silence, a dash or nothing for almost every line. A bounded fine-tuning of the last four decoder layers on 600 transcribed lines moves it off the floor but not far (@P:ADAPTED_CER@ CER and @P:ADAPTED_WER@ WER in the build record, @P:ADAPTED_EXACT@ of the held-out lines exact; the constant-transcript baseline scores 0.944) and the validation curve plateaus there at every rate tried, with a 269 MB adapter that reloads line-for-line. That is the claim: the adaptation contract works end to end on one task of a multi-task model with a real labelled set, and the numbers it produces are read as micro and macro rates, against two non-adapted baselines and the frozen model, with the hypothesis length beside them rather than in isolation — and they say that a decoder-only adapter cannot make this encoder read cursive. The obvious next experiment, adapting the BART encoder's last layers too (or the vision tower), is not in this repository.

The test split is 140 lines from one seeded draw of one 800-line sample, the validation split that picks the epoch is 60, and both rates are corpus edit distances over one crowdsourced transcription — not a benchmark, not a measure of reading order or layout. So a gain here says the contract works on one council's minutes, not that the adapted model handles other hands, other languages, other scripts or your scans. The decoder that was tuned serves every task token: the drawing re-run in Section 9 is one image of evidence about what the tuning did to `<CAPTION>` and `<OD>` (@P:DRAWING_AFTER@), not a measurement, and a deployment that needs the other tasks must measure them after adapting. The decoder was adapted, not the vision tower: what the encoder cannot resolve in a 128-px line squashed into a 768 × 768 square stays unread.

Three things to carry to real data. **Baselines first:** the empty and constant-transcript rates on *your* transcripts, and the frozen model's hypothesis length, are the numbers to read before any adapted one. **Rates above 1.0:** an uncapped CER tells you the model is generating, not reading; a capped one would hide it. **Leakage:** keep every image in one split (the contract de-duplicates by decoded pixels) and split by page, writer or volume when your lines come from few sources — lines cut from the same page share a hand.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled line set, validate the demonstrated dataset contract without leakage, execute the inference contract for three task tokens and a bounded fine-tuning of one of them with the model's own objective, evaluate against two non-adapted baselines and the frozen model on a line-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, recognition quality on any other hand, language or document family, caption or detection quality after adaptation, or production fitness.

**Optional experiments (they do not affect the default path):** set `LEARNING_RATE` to `1e-4` or `2e-4` with `EPOCHS = 8` and read the same plateau the build record found (0.798 and 0.807); raise `EPOCHS` and watch the validation CER pick the epoch; set `NUM_BEAMS` to `1` in Sections 6–9 and read what greedy decoding costs; lower `LINE_MAX_NEW_TOKENS` to `64` and read how the truncation count changes; add `<CAPTION_TO_PHRASE_GROUNDING>` to `CAPABILITIES` with the frozen caption as its `text_input`; or bring your own transcribed lines through BYOD and read the two baselines before the adapted number.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/florence-2-large-community/` and rerun Section 3. `RuntimeError: checkpoint does not match the native Florence-2 architecture` in Section 3: the staged weights are not the pinned converted checkpoint — re-stage. A "slow image processor" notice from `transformers` is expected and harmless.

## References

- Repository README: https://github.com/kurtvalcorza/florence2-vision-language-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/florence2-vision-language-pipeline/blob/main/MODEL_CARD.md
- Weight provenance and pin history: https://github.com/kurtvalcorza/florence2-vision-language-pipeline/blob/main/docs/WEIGHTS.md
- Pinned converted checkpoint: https://huggingface.co/florence-community/Florence-2-large
- Original weights and licence: https://huggingface.co/microsoft/Florence-2-large
- Florence-2 paper (Xiao et al., 2023): https://arxiv.org/abs/2311.06242
- Transformers Florence-2 documentation: https://huggingface.co/docs/transformers/model_doc/florence2
- Belfort-line dataset (Teklia, MIT): https://huggingface.co/datasets/Teklia/Belfort-line — Tarride et al., Handwritten Text Recognition from Crowdsourced Annotations (HIP 2023): https://doi.org/10.1145/3604951.3605517
- Sibling row on the same split: https://github.com/kurtvalcorza/got-ocr2-pipeline
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)